# RSNA Knee Abnormality Detection: Master End-to-End Framework

## 1. Executive Summary & Architecture Overview

This notebook implements an **end-to-end, multi-slot attention deep learning framework** for multi-target knee MRI abnormality detection on the RSNA competition dataset.

The system processes multi-sequence knee MRI studies by mapping acquisitions into 6 standardized anatomical slots, projecting DICOM slices along physical 3D space, extracting self-supervised visual features via a **DINOv2 backbone**, applying **Slot Attention with Presence Masking**, and ensembling predictions via **Percentile Rank Fusion**.

```
                         KNEE MRI STUDY (DICOM SERIES)
                                      │
              ┌───────────────────────┴───────────────────────┐
              ▼                                               ▼
   3D Spatial Normal Projection                    Laterality Normalization
  n = iop[:3] x iop[3:6], k = ipp.n               cx = ipp_x + center offset
              │                                               │
              └───────────────────────┬───────────────────────┘
                                      ▼
                        6-Slot Anatomical Router
       (SAG_FLUID_FS, COR_FLUID_FS, AX_FLUID_FS, SAG_FLUID_NOFS, COR_T1, SAG_T1)
                                      │
                                      ▼
                      Percentile Intensity Normalization
                            (1st - 99th Percentile)
                                      │
                                      ▼
                           DINOv2 Feature Extractor
                          (CLS + Mean + Focal Patch)
                                      │
                                      ▼
                       Slot Attention Network (SlotHead)
                       att = masked_fill(mask < 0.5, -1e4)
                                      │
                                      ▼
                     Multi-Window Spatial Sliding TTA
                          (Target-Specific Pooling)
                                      │
                                      ▼
                        Percentile Rank Fusion Ensemble
                                 submission.csv
```

---

### Core System Specifications

| Component | Technical Implementation | Rationale |
|---|---|---|
| **Backbone** | DINOv2 ViT-Small (`facebook/dinov2-small`) | Self-supervised visual representations for fine-grained tissue structures |
| **Spatial Projection** | Physical normal projection $k = \mathbf{ipp} \cdot \mathbf{n}$ | Eliminates arbitrary UID filename noise; orders slices along physical patient depth |
| **Laterality Normalization** | DICOM tag + Center-x ($\Delta x \ge 20\text{mm}$) | Normalizes Right knees to Left-knee orientation; prevents midline mis-flipping |
| **Anatomical Router** | 6-Slot plane & contrast classifier | Standardizes variable MRI protocol acquisitions across institutions |
| **Slot Attention** | Cross-attention with $-10^4$ presence penalty | Focuses model attention only on acquired sequences while penalizing missing slots |
| **Spatial TTA** | 4-10 sliding windows ($G=3$) + target pooling | Evaluates full joint depth; `max` for focal injuries, `top2` for ligaments |
| **Ensemble Fusion** | Percentile rank-mean aggregation | Scale-invariant rank fusion across heterogeneous checkpoint configurations |

---

### 12 Target Abnormality Definitions

| Abnormality Target | Category | Primary Slots | TTA Pooling Mode |
|---|---|---|---|
| **ACL** | Ligament Tear | SAG_FLUID_FS, SAG_FLUID_NOFS, SAG_T1 | `top2` |
| **MCL** | Ligament Tear | COR_FLUID_FS, COR_T1 | `top2` |
| **Medial Meniscus** | Meniscal Tear | SAG_FLUID_FS, COR_FLUID_FS, SAG_FLUID_NOFS, COR_T1 | `max` |
| **Lateral Meniscus** | Meniscal Tear | SAG_FLUID_FS, COR_FLUID_FS, SAG_FLUID_NOFS, COR_T1 | `max` |
| **Medial OA** | Osteoarthritis | COR_FLUID_FS, COR_T1, SAG_T1 | `mean` |
| **Lateral OA** | Osteoarthritis | COR_FLUID_FS, COR_T1, SAG_T1 | `mean` |
| **PF OA** | Osteoarthritis | SAG_FLUID_FS, AX_FLUID_FS, SAG_T1 | `mean` |
| **Effusion** | Fluid / Synovial | SAG_FLUID_FS, AX_FLUID_FS | `mean` |
| **Synovitis** | Fluid / Synovial | SAG_FLUID_FS, AX_FLUID_FS | `mean` |
| **Baker's** | Popliteal Cyst | SAG_FLUID_FS | `max` |
| **Contusion** | Bone Bruise | SAG_FLUID_FS, COR_FLUID_FS, AX_FLUID_FS | `max` |
| **Fracture** | Bone Injury | SAG_FLUID_FS, COR_FLUID_FS, AX_FLUID_FS, COR_T1, SAG_T1 | `max` |


In [ ]:
import os
import sys
import gc
import re
import math
import time
import json
import traceback
import unicodedata
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoConfig

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")


In [ ]:
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
N_SLOT = len(SLOTS)

SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

TTA_TARGET_POOL = {
    "Fracture": "max",
    "Contusion": "max",
    "Medial Meniscus": "max",
    "Lateral Meniscus": "max",
    "ACL": "top2",
    "MCL": "top2",
    "Baker's": "max",
}

POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}

RULES_NATIVE = {"order": "normal", "lat": "centre", "slot_fallback": False, "decode_fill": "nearest"}
RULES_LEGACY = {"order": "dominant_axis", "lat": "corner_x", "slot_fallback": True, "decode_fill": "zero"}

CROP_MM = 130.0
IMG_SIZE = 336
SLICE_BAND = (0.20, 0.80)
CACHE_SLICES = 6
GROUP = 3
N_SLICES_PER_SLOT = 6
BATCH_STUDIES = 2
NUM_WORKERS = 2
EPOCHS = 3
LR_BACKBONE = 8e-6
LR_HEAD = 1e-3
UNFREEZE_LAST = 6
WEIGHT_DECAY = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loaded {len(TARGETS)} targets & {N_SLOT} slots.")
print(f"Crop: {CROP_MM}mm | Resolution: {IMG_SIZE}x{IMG_SIZE} | SliceBand: {SLICE_BAND} | CacheSlices: {CACHE_SLICES} | Group: {GROUP}")
print(f"Device: {DEVICE}")


## 2. Multilingual Clinical NLP Pseudo-Label Parser

When unannotated training reports are available, the NLP parser extracts weak supervision labels across 12 targets from radiology text.

### Text Normalization & Negation Architecture

```
         Raw Radiology Report Text
                     │
                     ▼
         Unicode & Accent Stripping (NFKD)
                     │
                     ▼
        Hard-Wrap Line Join & Punctuation Standardizer
                     │
                     ▼
          Clause Segmentation ([.;!?\n])
                     │
                     ▼
    ┌────────────────┴────────────────┐
    ▼                                 ▼
Negation Detector                Regex Pathology Matcher
(English, Spanish, French,       (TEAR, OA, EFFUSION, SYNVI,
 German, Dutch, Turkish, etc.)    BAKER, CONTUSION, FRACTURE)
    │                                 │
    └────────────────┬────────────────┘
                     ▼
          Target Pseudo-Labels (0.0 / 1.0)
```

---

### Regex Vocabulary & Pathology Mapping

| Pathology | Core Keyword Regex | Multilingual Tokens Matched | Target Assigned |
|---|---|---|---|
| **Ligament / Meniscus Tear** | `\b(tear\|torn\|rupture\|disruption\|avuls\|rotura\|desgarro\|dechirure\|scheur\|yirtik)\b` | English, Spanish, French, German, Dutch, Turkish | ACL, MCL, Medial/Lateral Meniscus |
| **Osteoarthritis** | `(osteoarthrit\|arthros\|gonarthros\|chondropath\|chondromalac\|cartilage loss\|osteophyt)` | Clinical terms, Cartilage degradation, Osteophytes | Medial OA, Lateral OA, PF OA |
| **Joint Effusion** | `\b(effusion\|joint fluid\|intra ?articular fluid\|hydrops\|derrame\|epanchement\|gelenkerguss)\b` | Hydrops, Intra-articular fluid | Effusion |
| **Synovitis** | `(synovit\|sinovit\|synovial thickening\|hoffit)` | Synovium inflammation, Hoffa's disease | Synovitis |
| **Baker's Cyst** | `(baker\|popliteal cyst\|quiste popliteo\|kyste poplite\|bakerzyste)` | Popliteal cyst | Baker's |
| **Bone Contusion** | `\b(contusion\|bone bruise\|marrow edema\|marrow oedema\|edema oseo\|knochenmarkodem)\b` | Marrow edema, Bone bruise | Contusion |
| **Fracture** | `\b(fracture\|fract\|fractura\|fractuur\|fraktur\|kirik\|prijelom)\b` | Bone cortex disruption | Fracture |


In [ ]:
_PRE_TRANS = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})

def normalize_clinical_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE_TRANS).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("\u00ad", "")
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def unwrap_hard_wraps(text: str) -> str:
    if not isinstance(text, str):
        return ""
    lines = text.split("\n")
    out = []
    for line in lines:
        s = line.strip()
        if out and out[-1] and not re.search(r"[.;:!?>*]$", out[-1]) and len(out[-1].split()) >= 4 and s and not s[:1].isupper():
            out[-1] = out[-1] + " " + s
        else:
            out.append(s)
    return "\n".join(out)

PRE_NEGATION = re.compile(
    r"\b(no|not|without|negative for|absence|no evidence|free of|none|neither|nor|"
    r"sin|no hay|ausencia|ausentes?|no se|pas de|sans|aucune?|geen|zonder|niet|"
    r"keine?[nmrs]?|ohne|nicht|kein|nema|bez|nisu|nije)\b"
)

PATTERNS = {
    "TEAR": re.compile(r"\b(tear|torn|rupture|disruption|avuls|rotura|desgarro|roto|dechirure|scheur|ruptuur|riss|einriss|yirtik|puknuce)\b"),
    "OA": re.compile(r"(osteoarthrit|arthros|gonarthros|chondropath|chondromalac|condropat|condromalac|cartilage loss|osteophyt|osteofit|artrose|gonartrose)"),
    "EFFUSION": re.compile(r"\b(effusion|joint fluid|intra ?articular fluid|hydrops|derrame|liquido articular|epanchement|gewrichtsvocht|gelenkerguss|efuzyon|izljev)\b"),
    "SYNVI": re.compile(r"(synovit|sinovit|synovial thickening|hoffit)"),
    "BAKER": re.compile(r"(baker|popliteal cyst|quiste popliteo|kyste poplite|poplitealzyste|bakerzyste|bakerova)"),
    "CONTUSION": re.compile(r"\b(contusion|bone bruise|marrow edema|marrow oedema|edema oseo|botcontusie|knochenmarkodem|kemik odemi|kostani edem)\b"),
    "FRACTURE": re.compile(r"\b(fracture|fract|fractura|fractuur|fraktur|kirik|prijelom)\b")
}

def extract_labels(report_text: str) -> Dict[str, float]:
    cleaned = normalize_clinical_text(unwrap_hard_wraps(report_text))
    labels = {target: 0.0 for target in TARGETS}
    if not cleaned:
        return labels
    clauses = [c.strip() for c in re.split(r"[.;!?]\s+|\n+", cleaned) if c.strip()]
    for clause in clauses:
        is_negated = bool(PRE_NEGATION.search(clause))
        if PATTERNS["EFFUSION"].search(clause):
            labels["Effusion"] = 0.0 if is_negated else 1.0
        if PATTERNS["SYNVI"].search(clause):
            labels["Synovitis"] = 0.0 if is_negated else 1.0
        if PATTERNS["BAKER"].search(clause):
            labels["Baker's"] = 0.0 if is_negated else 1.0
        if PATTERNS["CONTUSION"].search(clause):
            labels["Contusion"] = 0.0 if is_negated else 1.0
        if PATTERNS["FRACTURE"].search(clause):
            labels["Fracture"] = 0.0 if is_negated else 1.0
        if PATTERNS["TEAR"].search(clause):
            if "acl" in clause or "anterior cruciate" in clause:
                labels["ACL"] = 0.0 if is_negated else 1.0
            if "mcl" in clause or "medial collateral" in clause:
                labels["MCL"] = 0.0 if is_negated else 1.0
            if "medial meniscus" in clause or "menisco medial" in clause:
                labels["Medial Meniscus"] = 0.0 if is_negated else 1.0
            if "lateral meniscus" in clause or "menisco lateral" in clause:
                labels["Lateral Meniscus"] = 0.0 if is_negated else 1.0
        if PATTERNS["OA"].search(clause):
            if "patellofemoral" in clause or "patelar" in clause or "trochlea" in clause or "retropatellar" in clause:
                labels["PF OA"] = 0.0 if is_negated else 1.0
            elif "medial" in clause:
                labels["Medial OA"] = 0.0 if is_negated else 1.0
            elif "lateral" in clause:
                labels["Lateral OA"] = 0.0 if is_negated else 1.0
            else:
                v = 0.0 if is_negated else 1.0
                labels["Medial OA"] = v
                labels["Lateral OA"] = v
    return labels

# Validation check
demo_sample = extract_labels("No ACL tear. Medial meniscus tear present. Small joint effusion.")
print("NLP Parser Validation Sample:")
for k, v in demo_sample.items():
    if v > 0:
        print(f"  - {k}: {v}")


## 3. DICOM 3D Spatial Geometry & Dataset Preprocessing

### 1. Physical 3D Spatial Slice Projection

In Kaggle test sets, DICOM filenames are assigned arbitrary SOP Instance UID hashes (e.g., `1.2.826.0...dcm`). Natural string sorting on UID strings produces a random permuted order ($r = 0.009$, zero correlation with spatial depth!).

To recover exact physical depth ordering, each slice's 3D spatial position vector $\mathbf{ipp} = (x, y, z)$ is projected onto the slice normal vector $\mathbf{n}$:

$$\mathbf{n} = \mathbf{iop}_{[:3]} \times \mathbf{iop}_{[3:6]}$$

$$k = \mathbf{ipp} \cdot \mathbf{n} = \text{ImagePositionPatient} \cdot (\text{ImageOrientationPatient}_{\text{row}} \times \text{ImageOrientationPatient}_{\text{col}})$$

Slices are sorted monotonically by $k$, ordering them strictly along the anatomical depth of the joint.

---

### 2. Laterality Normalization & Center-X Midline Safety Threshold

To map all knees onto a left-knee convention:
- **Coronal & Axial**: Horizontally flipped (`torch.flip(dim=-1)`).
- **Sagittal**: Slice sequence reversed (`torch.flip(dim=0)`).

If DICOM `Laterality` / `ImageLaterality` tags are absent, the patient x-coordinate is evaluated at the image **center**:

$$c_x = \text{ipp}_x + \text{iop}_{x,0} \cdot \text{ps}_y \cdot \frac{\text{cols}}{2} + \text{iop}_{x,1} \cdot \text{ps}_x \cdot \frac{\text{rows}}{2}$$

$$\text{Laterality} = \begin{cases} \text{Right ("R")}, & c_x \le -20.0\text{ mm} \\ \text{Left ("L")}, & c_x \ge +20.0\text{ mm} \\ \text{Unresolved (None)}, & |c_x| < 20.0\text{ mm} \end{cases}$$

A $20.0\text{ mm}$ midline dead zone prevents mis-flipping scans acquired near the patient midline.


In [ ]:
_SEP = re.compile(r"[\s_/\\]+")
_FATSAT_RX = re.compile(r"(fat\s*sat|fatsat|\bfs\b|spair|spir|tirm|stak)")
_T1_RX = re.compile(r"\bt1\b")
_T2_RX = re.compile(r"\bt2\b")
_PD_RX = re.compile(r"\b(pd|proton)\b")
FATSAT_OPTS = {"FS", "FAT SAT", "FATSAT", "SAT1", "SPAIR", "SPIR", "TIRM"}

def classify_series_meta(desc, seq_name, scan_opts, tr, te):
    full_desc = (str(desc) + " " + str(seq_name)).lower()
    opts = [t.strip().upper() for t in str(scan_opts).split("|")]
    fatsat = bool(_FATSAT_RX.search(full_desc)) or any(t in FATSAT_OPTS for t in opts)
    t1 = bool(_T1_RX.search(full_desc))
    t2 = bool(_T2_RX.search(full_desc))
    pdw = bool(_PD_RX.search(full_desc))
    tr_num = float(tr) if tr and str(tr).replace('.','',1).isdigit() else None
    te_num = float(te) if te and str(te).replace('.','',1).isdigit() else None
    if t1 and not t2 and not pdw: weight = "T1"
    elif t2 and not pdw: weight = "T2"
    elif pdw: weight = "PD"
    elif tr_num is not None and tr_num < 800: weight = "T1"
    elif te_num is not None and te_num > 60: weight = "T2"
    elif tr_num is not None and tr_num >= 800: weight = "PD"
    else: weight = "UNK"
    return weight in ["PD", "T2"], fatsat

def order_slices(files, order_rule="normal"):
    # Matches 0.899_code.py order_slices() exactly (lines 1431-1475)
    # Projects ImagePositionPatient onto slice normal = cross(iop[:3], iop[3:6])
    if not files:
        return files
    if order_rule == "dominant_axis":
        def _order_dom(p):
            try:
                ds = pydicom.dcmread(str(p), stop_before_pixels=True, force=True)
                iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
                ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
                normal = np.cross(iop[:3], iop[3:6])
                ax = np.argmax(np.abs(normal))
                return float(ipp[ax])
            except Exception:
                return 0.0
        return sorted(files, key=_order_dom)

    keyed = []
    for p in files:
        k = None
        try:
            ds = pydicom.dcmread(str(p), stop_before_pixels=True, force=True)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            normal = np.cross(iop[:3], iop[3:6])
            k = float(np.dot(ipp, normal))
        except Exception:
            try:
                ds = pydicom.dcmread(str(p), stop_before_pixels=True, force=True)
                k = float(getattr(ds, "InstanceNumber", 0))
            except Exception:
                k = None
        keyed.append((k, p))

    if any(k is None for k, _ in keyed):
        return sorted(files, key=lambda p: tuple(int(x) if x.isdigit() else x for x in re.split(r"(\d+)", p.name)))
    return [p for _, p in sorted(keyed, key=lambda t: t[0])]

def normalise_laterality(t, plane, lat):
    # t: (N_SLICE, H, W) tensor
    # Matches 0.899_code.py normalise_laterality() exactly
    if lat != "R":
        return t
    if plane in ("Coronal", "Axial"):
        return torch.flip(t, dims=[-1])
    return torch.flip(t, dims=[0])

def read_slot_slices(files, n_slice, out_size, px, plane=None, lat=None, slice_band=SLICE_BAND, order_rule="normal"):
    # Read n_slice slices spread over slice_band with percentile normalization.
    # Matches 0.899_code.py read_slot() exactly.
    if not files:
        return None
    files = order_slices(files, order_rule=order_rule)
    n = len(files)
    lo_idx = int(slice_band[0] * (n - 1))
    hi_idx = int(slice_band[1] * (n - 1))
    if hi_idx > lo_idx:
        indices = np.unique(np.linspace(lo_idx, hi_idx, n_slice).astype(int))
    else:
        indices = np.array([n // 2])
    while len(indices) < n_slice:
        indices = np.append(indices, indices[-1])
    indices = indices[:n_slice]

    planes = []
    for i in indices:
        try:
            ds = pydicom.dcmread(str(files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
            planes.append(a)
        except Exception:
            planes.append(None)

    got = [k for k, p in enumerate(planes) if p is not None]
    if not got:
        return None
    if len(got) < len(planes):
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0).float()
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False).squeeze(0)

    if plane and lat:
        t = normalise_laterality(t, plane, lat)

    return (t * 255).round().clamp(0, 255).to(torch.uint8)


class KneeSlotDataset(Dataset):
    def __init__(self, df, img_dir, series_meta_df=None, is_train=False,
                 n_slices=N_SLICES_PER_SLOT, img_size=IMG_SIZE, slice_band=SLICE_BAND, rules=None):
        self.df = df
        self.img_dir = Path(img_dir)
        self.is_train = is_train
        self.n_slices = n_slices
        self.img_size = img_size
        self.slice_band = slice_band
        self.rules = rules or RULES_NATIVE
        self.order_rule = self.rules.get("order", "normal")
        self.lat_rule = self.rules.get("lat", "centre")
        self.plane_map = {}
        if series_meta_df is not None and "SeriesInstanceUID" in series_meta_df.columns:
            if "Anatomical_Plane" in series_meta_df.columns:
                self.plane_map = dict(zip(
                    series_meta_df["SeriesInstanceUID"].astype(str),
                    series_meta_df["Anatomical_Plane"].astype(str)))

    def __len__(self):
        return len(self.df)

    def _get_series_plane(self, series_dir, slices):
        uid = series_dir.name
        if uid in self.plane_map:
            return self.plane_map[uid]
        if slices:
            try:
                dcm = pydicom.dcmread(str(slices[0]), stop_before_pixels=True)
                iop = [float(x) for x in getattr(dcm, 'ImageOrientationPatient', [])]
                if len(iop) >= 6:
                    normal = np.cross(iop[:3], iop[3:6])
                    ax = np.argmax(np.abs(normal))
                    return ["Sagittal", "Coronal", "Axial"][ax]
            except Exception:
                pass
        return "Sagittal"

    def _get_series_meta(self, slices):
        if not slices:
            return True, True, None, None
        try:
            dcm = pydicom.dcmread(str(slices[0]), stop_before_pixels=True)
            desc = getattr(dcm, 'SeriesDescription', '')
            seq_name = getattr(dcm, 'SequenceName', '')
            scan_opts = getattr(dcm, 'ScanOptions', '')
            tr = getattr(dcm, 'RepetitionTime', None)
            te = getattr(dcm, 'EchoTime', None)
            ps = getattr(dcm, 'PixelSpacing', None)
            px = float(ps[0]) if ps else None
            
            # Determine laterality: matches RULES_NATIVE lat="centre" (20mm threshold)
            lat = getattr(dcm, 'Laterality', None) or getattr(dcm, 'ImageLaterality', None)
            if lat:
                lat = str(lat).strip().upper()[0] if str(lat).strip() in ("L", "R") else None
            if not lat and self.lat_rule == "centre":
                ipp = getattr(dcm, 'ImagePositionPatient', None)
                iop = getattr(dcm, 'ImageOrientationPatient', None)
                rows = getattr(dcm, 'Rows', None)
                cols = getattr(dcm, 'Columns', None)
                if ipp and iop and ps and rows and cols and len(ipp) >= 3 and len(iop) >= 6:
                    cx = float(ipp[0]) + float(iop[0]) * float(ps[1]) * float(cols) / 2.0 + float(iop[3]) * float(ps[0]) * float(rows) / 2.0
                    if abs(cx) >= 20.0:
                        lat = "R" if cx < 0 else "L"

            fluid, fatsat = classify_series_meta(desc, seq_name, scan_opts, tr, te)
            return fluid, fatsat, px, lat
        except Exception:
            return True, True, None, None

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_uid = str(row['StudyInstanceUID'])
        study_dir = self.img_dir / study_uid

        # Output: (N_SLOT, n_slices, img_size, img_size) uint8-valued float
        slots_tensor = torch.zeros((N_SLOT, self.n_slices, self.img_size, self.img_size), dtype=torch.float32)
        mask = torch.zeros(N_SLOT, dtype=torch.float32)

        if study_dir.exists():
            series_dirs = [d for d in study_dir.iterdir() if d.is_dir()]
            series_info = []
            for s_dir in series_dirs:
                slices = list(s_dir.glob("*.dcm"))
                if slices:
                    plane = self._get_series_plane(s_dir, slices)
                    fluid, fatsat, px, lat = self._get_series_meta(slices)
                    series_info.append({
                        'dir': s_dir, 'slices': slices, 'n_slices': len(slices),
                        'plane': plane, 'fluid': fluid, 'fatsat': fatsat, 'px': px, 'lat': lat
                    })

            for slot_idx, (s_name, target_plane, target_fluid, target_fs) in enumerate(SLOTS):
                # Strict matching: plane + fatsat (+ fluid if specified)
                candidates = [s for s in series_info
                              if s['plane'] == target_plane
                              and s['fatsat'] == target_fs
                              and (target_fluid is None or s['fluid'] == target_fluid)]
                if candidates:
                    best = max(candidates, key=lambda c: c['n_slices'])
                    slot_data = read_slot_slices(
                        best['slices'], self.n_slices, self.img_size, best['px'],
                        plane=best['plane'], lat=best.get('lat'), slice_band=self.slice_band,
                        order_rule=self.order_rule)
                    if slot_data is not None:
                        slots_tensor[slot_idx] = slot_data.float()
                        mask[slot_idx] = 1.0

        if self.is_train:
            labels_tensor = torch.tensor([float(row.get(t, 0.0)) for t in TARGETS], dtype=torch.float32)
            return slots_tensor, mask, labels_tensor

        return slots_tensor, mask, study_uid

print("KneeSlotDataset with SLICE_BAND, percentile norm, RescaleSlope, and proper mask compiled.")


## 4. DINOv2 Backbone & Slot Attention Network Architecture

### 1. Backbone Representation

The visual encoder is initialized from **DINOv2 ViT-Small** (`facebook/dinov2-small`).
For each slice, the backbone outputs patch tokens $\mathbf{X}_{patch} \in \mathbb{R}^{B \times P \times D}$ ($P = 576$ for $336\times 336$) and CLS token $\mathbf{x}_{cls} \in \mathbb{R}^{B \times D}$.

Feature pooling combines:
1. CLS Token $\mathbf{x}_{cls}$
2. Mean Patch Vector $\frac{1}{P} \sum_{p=1}^P \mathbf{x}_p$
3. **Focal Top-k Patch Pool** (`cls_mean_focal`): Top $\lfloor P/8 \rfloor$ strongest patch responses per channel (retains localized pathology responses).

---

### 2. Slot Attention & Presence Masking (`SlotHead`)

$$\mathbf{H}_s = \text{GELU}\left(\text{LayerNorm}(\mathbf{X}_s) \mathbf{W}_{proj}\right) + \mathbf{E}_{slot}$$

$$\mathbf{A}_{o, s} = \frac{\mathbf{H}_s \cdot \mathbf{Q}_o}{\sqrt{d_{hidden}}} + \mathbf{P}_{prior}(o, s)$$

$$\mathbf{A}_{masked} = \text{masked\_fill}(\mathbf{A}, \text{Mask} < 0.5, -10^4) \longrightarrow \text{Softmax}(\mathbf{A}_{masked})$$

$$\mathbf{C}_o = \text{Dropout}\left(\sum_{s=1}^{S} \mathbf{A}_{o, s} \mathbf{H}_s\right)$$

$$\hat{\mathbf{y}}_o = \sum_{h=1}^{d_{hidden}} \mathbf{C}_{o, h} \mathbf{W}_{out, o, h} + b_{out, o}$$

The $-10^4$ presence penalty forces attention weights to **0.0 for absent slots**, preventing non-acquired series from corrupting predictions.


In [ ]:
class SlotHead(nn.Module):
    def __init__(self, dim, n_slot=N_SLOT, n_out=len(TARGETS), hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        self.prior = prior
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        # x: (B, S, dim), mask: (B, S) with 1=present, 0=absent.
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior and hasattr(self, "slot_prior"):
            att = att + self.slot_prior.unsqueeze(0)
        # Exact presence masking: line 1750 of 0.899_code.py
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class KneeDINOv2Model(nn.Module):
    def __init__(self, model_path_or_name, num_classes=len(TARGETS), n_slot=N_SLOT,
                 hidden_head=256, pool="cls_mean", prior=False, unfreeze_last=UNFREEZE_LAST):
        super().__init__()
        self.pool = pool
        self.backbone = AutoModel.from_pretrained(model_path_or_name)
        hidden_dim = self.backbone.config.hidden_size
        pooled_dim = hidden_dim * POOL_PARTS.get(pool, 2)

        self.head = SlotHead(dim=pooled_dim, n_slot=n_slot, n_out=num_classes,
                             hidden=hidden_head, prior=prior)

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        n_layers = len(self.backbone.encoder.layer)
        for p in self.backbone.parameters():
            p.requires_grad = False
        for blk in self.backbone.encoder.layer[max(0, n_layers - unfreeze_last):]:
            for p in blk.parameters():
                p.requires_grad = True
        if hasattr(self.backbone, 'layernorm') and self.backbone.layernorm is not None:
            for p in self.backbone.layernorm.parameters():
                p.requires_grad = True

    @staticmethod
    def detect_config_from_state_dict(state_dict):
        if "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        elif "model" in state_dict:
            state_dict = state_dict["model"]

        prior = "head.slot_prior" in state_dict
        n_slot, hidden = 6, 256
        if "head.slot_emb" in state_dict:
            n_slot, hidden = state_dict["head.slot_emb"].shape

        pool = "cls_mean"
        if "head.proj.1.weight" in state_dict:
            in_features = state_dict["head.proj.1.weight"].shape[1]
            if in_features > 800:
                pool = "cls_mean_focal"

        return {"pool": pool, "prior": prior, "n_slot": n_slot, "hidden": hidden}

    def adapt_and_load_state_dict(self, state_dict):
        if isinstance(state_dict, dict):
            if "state_dict" in state_dict:
                state_dict = state_dict["state_dict"]
            elif "model" in state_dict:
                state_dict = state_dict["model"]

        cfg = self.detect_config_from_state_dict(state_dict)
        hidden_dim = self.backbone.config.hidden_size
        pooled_dim = hidden_dim * POOL_PARTS.get(cfg["pool"], 2)

        needs_rebuild = False
        if cfg["pool"] != self.pool:
            self.pool = cfg["pool"]
            needs_rebuild = True
        if self.head.slot_emb.shape[0] != cfg["n_slot"] or self.head.hidden != cfg["hidden"]:
            needs_rebuild = True
        if cfg["prior"] != self.head.prior:
            needs_rebuild = True

        if needs_rebuild:
            self.head = SlotHead(dim=pooled_dim, n_slot=cfg["n_slot"], n_out=len(TARGETS),
                                 hidden=cfg["hidden"], prior=cfg["prior"]).to(DEVICE)

        self.load_state_dict(state_dict, strict=False)

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std

        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

print("KneeDINOv2Model with per-member config auto-detection compiled.")


## 5. Multi-Dataset Checkpoint Scanner

Scans attached Kaggle input paths for manifest-driven weights packages and standalone checkpoint files.


In [ ]:
def find_competition_root(input_root="/kaggle/input"):
    root = Path(input_root)
    if not root.exists():
        return root / "rsna-knee-abnormality-detection"
    for dirpath, dirnames, filenames in os.walk(root):
        if "test.csv" in filenames or "train.csv" in filenames:
            print(f"[Discovery] Competition root: {dirpath}")
            return Path(dirpath)
    return root / "rsna-knee-abnormality-detection"

def find_manifest_members(input_root="/kaggle/input"):
    root_path = Path(input_root)
    members = []
    if not root_path.exists():
        return members
    for dirpath, dirnames, filenames in os.walk(root_path):
        if "competitions" in dirpath or "models/metaresearch" in dirpath:
            continue
        if "manifest.json" in filenames:
            m_path = Path(dirpath) / "manifest.json"
            try:
                data = json.loads(m_path.read_text())
                for m in data.get("members", []):
                    w_file = m.get("file")
                    w_path = Path(dirpath) / w_file
                    if w_path.exists():
                        pg_raw = m.get("pixel_group", "")
                        pg_cfg = json.loads(pg_raw) if pg_raw and pg_raw.startswith("{") else {}
                        merged_cfg = {**m.get("config", {}), **pg_cfg}
                        print(f"[Discovery] Manifest member: {w_file}")
                        members.append({"path": w_path, "config": merged_cfg, "source": "manifest", "file": w_file})
            except Exception:
                pass
    return members

def find_standalone_checkpoints(input_root="/kaggle/input"):
    root_path = Path(input_root)
    ckps = []
    if not root_path.exists():
        return ckps
    for dirpath, dirnames, filenames in os.walk(root_path):
        if "competitions" in dirpath or "models/metaresearch" in dirpath:
            continue
        for f in filenames:
            if f.endswith(".pt") and f != "manifest.json":
                w_path = Path(dirpath) / f
                print(f"[Discovery] Standalone checkpoint: {f}")
                ckps.append({"path": w_path, "config": {}, "source": "standalone", "file": f})
    return ckps

def find_dinov2_base(input_root="/kaggle/input"):
    root_path = Path(input_root)
    if not root_path.exists():
        return None
    for dirpath, dirnames, filenames in os.walk(root_path):
        if "config.json" in filenames and ("dinov2" in dirpath.lower() or "small" in dirpath.lower()):
            return dirpath
    return None

comp_root = find_competition_root()
manifest_members = find_manifest_members()
standalone_ckps = find_standalone_checkpoints()
all_ckps = manifest_members + standalone_ckps

print(f"Competition Root: {comp_root}")
print(f"Total checkpoints: {len(all_ckps)} ({len(manifest_members)} manifest + {len(standalone_ckps)} standalone)")


## 6. End-to-End Training & Distillation Engine

When no pre-trained weights are present, the pipeline executes end-to-end training using OneCycleLR scheduling, binary cross-entropy loss with NLP soft labels, and report-grouped holdout validation.


In [ ]:
def macro_auc(y_true, y_pred):
    from sklearn.metrics import roc_auc_score
    aucs = []
    for j in range(y_true.shape[1]):
        if len(np.unique(y_true[:, j])) > 1:
            try:
                aucs.append(roc_auc_score(y_true[:, j], y_pred[:, j]))
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else 0.5

def train_model_end_to_end(train_csv_path, train_series_dir, save_path="trained_model.pt"):
    df = pd.read_csv(train_csv_path)
    if "Report" in df.columns:
        print("[Train] Extracting NLP labels from clinical reports...")
        nlp_labels = []
        for r in df["Report"].fillna(""):
            nlp_labels.append(extract_labels(r))
        nlp_df = pd.DataFrame(nlp_labels)
        for col in TARGETS:
            if col not in df.columns or df[col].isna().all():
                df[col] = nlp_df[col]
            else:
                df[col] = df[col].fillna(nlp_df[col])

    df = df.dropna(subset=["StudyInstanceUID"]).reset_index(drop=True)
    for col in TARGETS:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = df[col].fillna(0.0)

    import hashlib
    rep = df["Report"].fillna("") if "Report" in df.columns else df["StudyInstanceUID"]
    grp = np.array([int(hashlib.md5(str(s).encode()).hexdigest()[:8], 16) % 5 for s in rep])
    val_idx = np.where(grp == 0)[0]
    tr_idx = np.where(grp != 0)[0]

    tr_df, val_df = df.iloc[tr_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)
    print(f"[Train] Split: {len(tr_df)} train, {len(val_df)} holdout studies.")

    tr_ds = KneeSlotDataset(tr_df, train_series_dir, is_train=True)
    val_ds = KneeSlotDataset(val_df, train_series_dir, is_train=True)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_STUDIES, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_STUDIES, shuffle=False, num_workers=NUM_WORKERS)

    base_source = find_dinov2_base() or "facebook/dinov2-small"
    model = KneeDINOv2Model(base_source, pool="cls_mean_focal", prior=True).to(DEVICE)

    opt = torch.optim.AdamW([
        {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": LR_BACKBONE},
        {"params": model.head.parameters(), "lr": LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    total_steps = max(EPOCHS * len(tr_loader), 1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD], total_steps=total_steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

    best_auc, best_state = 0.0, None
    for ep in range(EPOCHS):
        model.train()
        tot_loss = 0.0
        for inputs, masks, labels in tr_loader:
            inputs, masks, labels = inputs.to(DEVICE), masks.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = model(inputs, masks)
                loss = F.binary_cross_entropy_with_logits(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot_loss += loss.item()

        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for inputs, masks, labels in val_loader:
                inputs, masks = inputs.to(DEVICE), masks.to(DEVICE)
                with torch.autocast("cuda", enabled=DEVICE.type == "cuda"):
                    probs = torch.sigmoid(model(inputs, masks))
                val_preds.append(probs.cpu().numpy())
                val_targets.append(labels.numpy())

        y_p = np.vstack(val_preds)
        y_t = np.vstack(val_targets)
        val_auc = macro_auc(y_t, y_p)
        print(f"[Epoch {ep+1}/{EPOCHS}] Loss: {tot_loss/len(tr_loader):.4f} | Holdout Macro AUC: {val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        torch.save({"model": best_state, "auc": best_auc}, save_path)
        print(f"[Train] Saved best model (AUC {best_auc:.4f}) to {save_path}")
        return save_path
    return None

print("Training engine compiled.")


## 7. Multi-Window Spatial TTA & Percentile Rank Fusion

### 1. Spatial Sliding TTA Indexing

For each study, the dataset extracts $N_{slices}$ slices (e.g. 12 slices) over `SLICE_BAND = (0.20, 0.80)`.
Sliding windows of size $G=3$ evaluate predictions over all depth segments:

$$\text{starts} = \{0, 1, 2, \dots, N_{slices} - G\}$$

---

### 2. Target-Specific TTA Pooling (`TTA_TARGET_POOL`)

$$\hat{y}(s, t) = \begin{cases} \max_{w} P_w(s, t), & t \in \{\text{Fracture, Contusion, Meniscus, Baker's}\} \\ \text{mean}\left(\text{Top2}_w P_w(s, t)\right), & t \in \{\text{ACL, MCL}\} \\ \frac{1}{W} \sum_{w=1}^W P_w(s, t), & \text{Otherwise} \end{cases}$$

---

### 3. Percentile Rank-Mean Ensemble Fusion

For an ensemble of $M$ models, raw probabilities $\hat{\mathbf{Y}}_m \in \mathbb{R}^{N \times 12}$ are converted into percentile ranks before averaging:

$$\mathbf{R}_m(s, t) = \text{rank}_{\text{pct}}\left(\hat{\mathbf{Y}}_m(s, t)\right) \in [0, 1]$$

$$\bar{\mathbf{R}}(s, t) = \frac{1}{M} \sum_{m=1}^M \mathbf{R}_m(s, t)$$

Percentile rank averaging prevents differently calibrated checkpoints from dominating the ensemble votes.


In [ ]:
def predict_member_tta(model, test_loader, device, n_slice=CACHE_SLICES, group=GROUP, target_img_size=IMG_SIZE):
    starts = list(range(n_slice - group + 1)) if n_slice >= group else [0]
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    model.eval()
    all_preds = []

    with torch.no_grad():
        for inputs, masks, _ in test_loader:
            masks = masks.to(device)
            win_probs = []

            for st in starts:
                sub_inputs = inputs[:, :, st:st + group].to(device)
                if sub_inputs.shape[-1] != target_img_size:
                    B, S, G, H, W = sub_inputs.shape
                    sub_flat = sub_inputs.view(B * S * G, 1, H, W)
                    sub_flat = F.interpolate(sub_flat, size=(target_img_size, target_img_size), mode="bilinear", align_corners=False)
                    sub_inputs = sub_flat.view(B, S, G, target_img_size, target_img_size)

                with torch.autocast("cuda", enabled=device.type == "cuda"):
                    logits = model(sub_inputs, masks, img_size=target_img_size).float()
                probs = torch.sigmoid(logits)
                win_probs.append(probs)

            win_stack = torch.stack(win_probs, dim=0)
            v = win_stack.mean(dim=0)

            if TTA_TARGET_POOL:
                for target, mode in TTA_TARGET_POOL.items():
                    if target in target_idx:
                        j = target_idx[target]
                        if mode == "max":
                            v[:, j] = win_stack[:, :, j].max(dim=0).values
                        elif mode in ("top2", "top3"):
                            k = min(int(mode[3:]), win_stack.shape[0])
                            v[:, j] = win_stack[:, :, j].topk(k, dim=0).values.mean(dim=0)

            all_preds.append(v.cpu().numpy())

    return np.vstack(all_preds) if all_preds else np.zeros((0, len(TARGETS)), np.float32)


def execute_master_pipeline():
    print("\n=========================================================================")
    print("    RSNA KNEE: MASTER END-TO-END PIPELINE                                ")
    print("=========================================================================")

    comp_root = find_competition_root()
    test_csv = comp_root / "test.csv"
    test_series_dir = comp_root / "test_series"
    test_series_csv = comp_root / "test_series.csv"
    train_csv = comp_root / "train.csv"
    train_series_dir = comp_root / "train_series"

    series_meta_df = pd.read_csv(test_series_csv) if test_series_csv.exists() else None

    if not test_csv.exists():
        test_df = pd.DataFrame({"StudyInstanceUID": ["demo_1", "demo_2", "demo_3"]})
    else:
        test_df = pd.read_csv(test_csv)
        print(f"[Pipeline] {len(test_df)} test studies.")

    weight_entries = find_manifest_members() + find_standalone_checkpoints()
    if not weight_entries and train_csv.exists():
        print("[Pipeline] No checkpoints found. Training end-to-end...")
        trained = train_model_end_to_end(train_csv, train_series_dir)
        if trained and Path(trained).exists():
            weight_entries.append({"path": Path(trained), "config": {}, "source": "trained"})

    base_source = find_dinov2_base() or "facebook/dinov2-small"
    per_member = []

    if not weight_entries:
        print("[Pipeline] No weights. Using baseline 0.5.")
        per_member.append(np.full((len(test_df), len(TARGETS)), 0.5, dtype=np.float32))
    else:
        print(f"[Pipeline] Evaluating {len(weight_entries)} checkpoint(s)...")

        groups = {}
        for entry in weight_entries:
            m_cfg = entry.get("config", {})
            key = (
                int(m_cfg.get("img", IMG_SIZE)),
                int(m_cfg.get("slices", CACHE_SLICES)),
                int(m_cfg.get("group", GROUP)),
                tuple(float(x) for x in m_cfg.get("band", SLICE_BAND))
            )
            groups.setdefault(key, []).append(entry)

        for gi, ((img_sz, n_slc, grp, s_band), g_entries) in enumerate(groups.items(), 1):
            print(f"\n[Decode Group {gi}/{len(groups)}] {img_sz}px x {n_slc} slices, band {s_band} -> {len(g_entries)} member(s)")
            group_rules = g_entries[0].get("config", {}).get("rules") or RULES_NATIVE
            group_ds = KneeSlotDataset(
                test_df, test_series_dir, series_meta_df, is_train=False,
                n_slices=n_slc, img_size=img_sz, slice_band=s_band, rules=group_rules
            )
            group_loader = DataLoader(group_ds, batch_size=BATCH_STUDIES, shuffle=False, num_workers=NUM_WORKERS)

            for idx, entry in enumerate(g_entries, 1):
                w_path = entry["path"]
                print(f"  [Pass {idx}/{len(g_entries)}] {w_path.name}")
                try:
                    state_dict = torch.load(w_path, map_location="cpu", weights_only=False)
                    cfg = KneeDINOv2Model.detect_config_from_state_dict(state_dict)
                    manifest_cfg = entry.get("config", {})
                    pool = manifest_cfg.get("pool", cfg["pool"])
                    prior = bool(manifest_cfg.get("prior", cfg["prior"]))
                    print(f"    Config: pool={pool}, prior={prior}, n_slot={cfg['n_slot']}, hidden={cfg['hidden']}")

                    model = KneeDINOv2Model(
                        base_source, pool=pool, prior=prior,
                        n_slot=cfg["n_slot"], hidden_head=cfg["hidden"]
                    ).to(DEVICE)
                    model.adapt_and_load_state_dict(state_dict)
                    model.eval()

                    preds = predict_member_tta(
                        model, group_loader, DEVICE,
                        n_slice=n_slc, group=grp, target_img_size=img_sz
                    )
                    per_member.append(preds)
                    print(f"  [Pass {idx}] Complete. Shape: {preds.shape}")

                    del model, state_dict
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                except Exception as err:
                    print(f"  [Pass {idx}] ERROR: {err}")

    n_samples = len(test_df)
    n_models = len(per_member)

    if n_models == 0:
        final = np.full((n_samples, len(TARGETS)), 0.5, dtype=np.float32)
    elif n_models == 1:
        final = per_member[0]
    else:
        print(f"\n[Ensemble] Rank fusion across {n_models} members...")
        acc = np.zeros((n_samples, len(TARGETS)), np.float64)
        for m_pred in per_member:
            r = pd.DataFrame(m_pred).rank(pct=True).to_numpy()
            acc += r
        final = (acc / n_models).astype(np.float32)

    submission_df = pd.DataFrame({"StudyInstanceUID": test_df["StudyInstanceUID"].values})
    for t_idx, target in enumerate(TARGETS):
        submission_df[target] = final[:, t_idx]

    submission_df.to_csv("submission.csv", index=False)
    print(f"\n=== SUCCESS: submission.csv ({submission_df.shape}) ===")
    print(submission_df.head(3).to_string(index=False))
    return submission_df


In [ ]:
sub = execute_master_pipeline()
